# Operations Research in Poland
## Facility Location, Exact Metric TSP, Lazy Row Generation, and Christofides Approximation

### Business context

The new fast-food chain **Steamboat Willie's** plans to expand into Poland.

This notebook answers three connected Operations Research questions using the supplied geographic dataset.

### Task 1 — Store location / covering problem

> What is the smallest number of store locations needed so that every populated place in the Polish subset of the dataset has a Steamboat Willie's within a 50 km radius?

We formulate this as a **minimum set-covering / facility-location problem**.

### Task 2 — Exact Metric Traveling Salesman Problem

After choosing store locations, suppose an inspection or logistics team must visit store locations.

We implement two exact symmetric metric TSP formulations:

1. **Explicit subtour-elimination ILP** with an exponential family of constraints on small benchmark instances.
2. **Lazy row generation**, where subtour constraints are added only when violated.

### Task 3 — Christofides approximation

We implement **Christofides' algorithm** and compare its tour length with the exact ILP optimum from Task 2.

### Important modeling interpretation

The dataset contains populated-place coordinates. Therefore:

- demand points = Polish populated places in the supplied dataset,
- candidate store locations = those same populated places,
- coverage distance = great-circle distance,
- TSP distance = great-circle distance.

This is an analytical location model, not a complete commercial site-selection study.


# Mathematical Formulation — Task 1

Let:

- $I$ = set of Polish populated places that must be covered,
- $J$ = candidate store locations,
- $d_{ij}$ = great-circle distance from demand point $i$ to candidate $j$,
- $R = 50$ km.

Define:

$$a_{ij}=
\begin{cases}
1, & d_{ij}\le 50\\
0, & \text{otherwise}
\end{cases}
$$

and binary decision variable:

$$x_j=
\begin{cases}
1, & \text{if a store is opened at candidate }j\\
0, & \text{otherwise.}
\end{cases}
$$

The minimum covering model is:

$$\min \sum_{j\in J}x_j$$

subject to:

$$\sum_{j\in J}a_{ij}x_j\ge 1
\qquad \forall i\in I$$

$$x_j\in\{0,1\}$$

If the MIP solver proves optimality, the result is the exact minimum for this model.

If the time limit is reached first, the notebook reports the best feasible solution plus the MIP lower bound and optimality gap rather than falsely claiming optimality.


# Mathematical Formulation — Task 2

For a symmetric TSP instance with node set $V$ and undirected edge set $E$, define:

$$x_{ij}\in\{0,1\}$$

for each edge $(i,j)$.

The objective is:

$$\min \sum_{(i,j)\in E}d_{ij}x_{ij}$$

Every node must have degree two:

$$\sum_{j:(i,j)\in E}x_{ij}=2
\qquad \forall i\in V$$

Degree constraints alone can create several disconnected cycles.

For every nontrivial subset $S\subset V$, the subtour-elimination constraint is:

$$\sum_{\substack{i<j\\i,j\in S}}x_{ij}\le |S|-1$$

There are exponentially many such subsets.

### Two exact strategies

**Explicit formulation:** generate the entire required family of subtour constraints for a small instance.

**Lazy row generation:** initially solve only the degree model, detect disconnected subtours, add only the violated subtour constraints, and resolve until a single Hamiltonian cycle remains.


# Mathematical Formulation — Task 3

Christofides' algorithm for a metric symmetric TSP:

1. Compute a minimum spanning tree.
2. Find all odd-degree vertices of the tree.
3. Compute a minimum-weight perfect matching on those odd vertices.
4. Combine tree and matching to obtain an Eulerian multigraph.
5. Find an Euler circuit.
6. Shortcut repeated vertices.

For a metric TSP:

$$L_{\text{Christofides}}\le \frac{3}{2}L^*$$

where $L^*$ is the optimal TSP tour length.

The notebook measures the observed approximation ratio:

$$\text{Ratio}
=
\frac{L_{\text{Christofides}}}{L^*}$$


In [ ]:
# Cell 1 — Imports

import os
import re
import json
import time
import math
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import BallTree
from sklearn.metrics.pairwise import haversine_distances

from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix, csr_matrix, vstack

import networkx as nx

warnings.filterwarnings("ignore")
np.random.seed(42)

print("Environment ready.")


In [ ]:
# Cell 2 — Reproducibility and model settings

EARTH_RADIUS_KM = 6371.0088
COVER_RADIUS_KM = 50.0

# Facility-location MIP settings.
# Increase this in Kaggle if you want a stronger optimality certificate.
FACILITY_TIME_LIMIT_SECONDS = 120
FACILITY_MIP_GAP_TARGET = 0.0

# TSP settings.
TSP_TIME_LIMIT_SECONDS = 120
TSP_MIP_GAP_TARGET = 0.0

# Explicit exponential subtour formulation is intentionally limited
# to small instances because the number of constraints grows exponentially.
EXPLICIT_TSP_MAX_N = 14

OUTPUT_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path(".")
)

print("Output folder:", OUTPUT_DIR.resolve())


In [ ]:
# Cell 3 — Find dataset.json in Kaggle or current environment

def locate_dataset():
    candidates = []

    # Local / notebook working directory
    for path in [
        Path("dataset.json"),
        Path("/mnt/data/dataset.json")
    ]:
        if path.exists():
            candidates.append(path)

    # Kaggle Add Input
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(kaggle_root.rglob("dataset.json"))

    if not candidates:
        raise FileNotFoundError(
            "dataset.json was not found. "
            "Upload the supplied JSON file to Kaggle using Add Input."
        )

    return candidates[0]


DATASET_PATH = locate_dataset()

print("Using:", DATASET_PATH)


In [ ]:
# Cell 4 — Load the supplied JSON

with open(DATASET_PATH, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print("Total records in supplied dataset:", len(raw_data))
print("Example keys:", list(raw_data[0].keys()))


In [ ]:
# Cell 5 — Filter Poland and create a clean table

records = []

for item in raw_data:
    if item.get("country_code") != "PL":
        continue

    coordinates = item.get("coordinates") or {}

    lat = coordinates.get("lat")
    lon = coordinates.get("lon")

    if lat is None or lon is None:
        continue

    records.append({
        "geoname_id": item.get("geoname_id"),
        "name": item.get("name"),
        "ascii_name": item.get("ascii_name"),
        "feature_class": item.get("feature_class"),
        "feature_code": item.get("feature_code"),
        "population": item.get("population"),
        "admin1_code": item.get("admin1_code"),
        "admin2_code": item.get("admin2_code"),
        "latitude": float(lat),
        "longitude": float(lon)
    })


poland = pd.DataFrame(records)

# The supplied Polish records are populated-place features.
poland = poland[
    poland["feature_class"].eq("P")
].copy()

poland = (
    poland
    .drop_duplicates(subset=["geoname_id"])
    .reset_index(drop=True)
)

print("Polish populated places with coordinates:", len(poland))
print("Latitude range:", poland["latitude"].min(), "to", poland["latitude"].max())
print("Longitude range:", poland["longitude"].min(), "to", poland["longitude"].max())

display(poland.head())


In [ ]:
# Cell 6 — Basic data-quality checks

assert len(poland) > 0
assert poland["latitude"].between(-90, 90).all()
assert poland["longitude"].between(-180, 180).all()
assert poland["geoname_id"].notna().all()

print("Missing population values:", poland["population"].isna().sum())
print("Unique feature codes:", poland["feature_code"].nunique())
print("Unique admin1 regions:", poland["admin1_code"].nunique())

display(
    poland["feature_code"]
    .value_counts()
    .head(15)
    .to_frame("count")
)


In [ ]:
# Cell 7 — Plot supplied Polish populated places

plt.figure(figsize=(9, 8))

plt.scatter(
    poland["longitude"],
    poland["latitude"],
    s=8,
    alpha=0.45
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(
    "Polish Populated Places in the Supplied Dataset"
)
plt.tight_layout()
plt.show()


# TASK 1 — Minimum Number of Stores for 50 km Coverage

We use great-circle distance rather than treating latitude and longitude as Cartesian coordinates.

For computational efficiency, a Haversine `BallTree` finds all candidate locations within 50 km of every demand point.


In [ ]:
# Cell 8 — Build the 50 km coverage neighborhoods

lat_lon_deg = poland[
    ["latitude", "longitude"]
].to_numpy(dtype=float)

lat_lon_rad = np.radians(lat_lon_deg)

tree = BallTree(
    lat_lon_rad,
    metric="haversine"
)

radius_radians = (
    COVER_RADIUS_KM
    / EARTH_RADIUS_KM
)

coverage_neighbors = tree.query_radius(
    lat_lon_rad,
    r=radius_radians
)

coverage_counts = np.array(
    [len(indices) for indices in coverage_neighbors],
    dtype=int
)

print("Demand points / candidates:", len(poland))
print("Minimum candidates within 50 km:", coverage_counts.min())
print("Median candidates within 50 km:", np.median(coverage_counts))
print("Mean candidates within 50 km:", coverage_counts.mean())
print("Maximum candidates within 50 km:", coverage_counts.max())


In [ ]:
# Cell 9 — Build sparse set-cover matrix

row_indices = []
column_indices = []

for demand_index, candidate_indices in enumerate(coverage_neighbors):
    row_indices.extend(
        [demand_index] * len(candidate_indices)
    )
    column_indices.extend(
        candidate_indices.tolist()
    )

coverage_matrix = coo_matrix(
    (
        np.ones(len(row_indices), dtype=float),
        (row_indices, column_indices)
    ),
    shape=(len(poland), len(poland))
).tocsr()

print("Coverage matrix shape:", coverage_matrix.shape)
print("Nonzero coverage relationships:", coverage_matrix.nnz)


In [ ]:
# Cell 10 — Greedy set-cover solution for a fast feasible upper bound

def greedy_set_cover(neighborhoods, n_points):
    cover_sets = [
        set(map(int, arr))
        for arr in neighborhoods
    ]

    uncovered = set(range(n_points))
    selected = []

    while uncovered:
        best_candidate = None
        best_gain = set()

        for candidate, covered in enumerate(cover_sets):
            gain = covered & uncovered

            if len(gain) > len(best_gain):
                best_candidate = candidate
                best_gain = gain

        if best_candidate is None or not best_gain:
            raise RuntimeError(
                "Greedy algorithm could not cover every demand point."
            )

        selected.append(best_candidate)
        uncovered -= best_gain

    # Remove redundant selected facilities.
    improved = selected.copy()

    changed = True
    while changed:
        changed = False

        for candidate in improved.copy():
            trial = [
                c for c in improved
                if c != candidate
            ]

            covered = set()
            for c in trial:
                covered |= cover_sets[c]

            if len(covered) == n_points:
                improved = trial
                changed = True
                break

    return improved


greedy_selected = greedy_set_cover(
    coverage_neighbors,
    len(poland)
)

print(
    "Greedy feasible store count:",
    len(greedy_selected)
)


In [ ]:
# Cell 11 — Exact / best-feasible minimum set-cover MIP

n_points = len(poland)

objective = np.ones(
    n_points,
    dtype=float
)

integrality = np.ones(
    n_points,
    dtype=int
)

bounds = Bounds(
    np.zeros(n_points),
    np.ones(n_points)
)

coverage_constraint = LinearConstraint(
    coverage_matrix,
    lb=np.ones(n_points),
    ub=np.full(n_points, np.inf)
)

facility_start_time = time.perf_counter()

facility_result = milp(
    c=objective,
    integrality=integrality,
    bounds=bounds,
    constraints=coverage_constraint,
    options={
        "time_limit": FACILITY_TIME_LIMIT_SECONDS,
        "mip_rel_gap": FACILITY_MIP_GAP_TARGET,
        "presolve": True
    }
)

facility_runtime = (
    time.perf_counter()
    - facility_start_time
)

print("Solver status:", facility_result.message)
print("Runtime (seconds):", round(facility_runtime, 3))

if facility_result.x is not None:
    mip_selected = np.where(
        facility_result.x > 0.5
    )[0].tolist()

    print(
        "Best MIP feasible store count:",
        len(mip_selected)
    )
else:
    mip_selected = None
    print("No MIP incumbent was returned.")


In [ ]:
# Cell 12 — Select the best available coverage solution

# Prefer the MIP incumbent if it improves on the greedy solution.
if mip_selected is not None and len(mip_selected) <= len(greedy_selected):
    selected_store_indices = mip_selected
    facility_solution_source = "MIP incumbent"
else:
    selected_store_indices = greedy_selected
    facility_solution_source = "Greedy feasible solution"

selected_stores = (
    poland
    .iloc[selected_store_indices]
    .copy()
    .reset_index()
    .rename(columns={"index": "original_index"})
)

facility_lower_bound = getattr(
    facility_result,
    "mip_dual_bound",
    np.nan
)

facility_gap = getattr(
    facility_result,
    "mip_gap",
    np.nan
)

facility_proven_optimal = bool(
    facility_result.success
    and facility_result.x is not None
)

print("Solution used:", facility_solution_source)
print("Stores opened:", len(selected_stores))
print("Solver proved optimal:", facility_proven_optimal)
print("MIP lower bound:", facility_lower_bound)
print("MIP relative gap:", facility_gap)

display(
    selected_stores[
        [
            "name",
            "population",
            "latitude",
            "longitude"
        ]
    ].head(20)
)


In [ ]:
# Cell 13 — Validate that every Polish point is covered within 50 km

selected_rad = np.radians(
    selected_stores[
        ["latitude", "longitude"]
    ].to_numpy(dtype=float)
)

selected_tree = BallTree(
    selected_rad,
    metric="haversine"
)

nearest_distance_rad, nearest_store_local_index = (
    selected_tree.query(
        lat_lon_rad,
        k=1
    )
)

nearest_distance_km = (
    nearest_distance_rad.ravel()
    * EARTH_RADIUS_KM
)

nearest_store_local_index = (
    nearest_store_local_index.ravel()
)

coverage_validation = poland[
    [
        "geoname_id",
        "name",
        "population",
        "latitude",
        "longitude"
    ]
].copy()

coverage_validation["nearest_store_index"] = (
    nearest_store_local_index
)

coverage_validation["nearest_store_name"] = [
    selected_stores.iloc[i]["name"]
    for i in nearest_store_local_index
]

coverage_validation["nearest_store_distance_km"] = (
    nearest_distance_km
)

coverage_validation["covered_within_50km"] = (
    nearest_distance_km
    <= COVER_RADIUS_KM + 1e-8
)

print(
    "All towns covered:",
    bool(
        coverage_validation[
            "covered_within_50km"
        ].all()
    )
)

print(
    "Maximum nearest-store distance:",
    round(
        coverage_validation[
            "nearest_store_distance_km"
        ].max(),
        4
    ),
    "km"
)

print(
    "Mean nearest-store distance:",
    round(
        coverage_validation[
            "nearest_store_distance_km"
        ].mean(),
        4
    ),
    "km"
)


In [ ]:
# Cell 14 — Plot towns and selected stores

plt.figure(figsize=(9, 8))

plt.scatter(
    poland["longitude"],
    poland["latitude"],
    s=7,
    alpha=0.25,
    label="Polish populated places"
)

plt.scatter(
    selected_stores["longitude"],
    selected_stores["latitude"],
    s=45,
    marker="x",
    label="Selected store locations"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(
    f"50 km Store-Covering Solution — "
    f"{len(selected_stores)} Stores"
)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cell 15 — Coverage-distance distribution

plt.figure(figsize=(9, 4))

plt.hist(
    coverage_validation[
        "nearest_store_distance_km"
    ],
    bins=30
)

plt.axvline(
    COVER_RADIUS_KM,
    linestyle="--",
    label="50 km requirement"
)

plt.xlabel("Distance to nearest selected store (km)")
plt.ylabel("Number of populated places")
plt.title("Coverage Quality")
plt.legend()
plt.tight_layout()
plt.show()


# Task 1 Interpretation

The notebook distinguishes between:

### Exact answer

If `facility_proven_optimal == True`, the selected store count is the exact optimum of the supplied-data model.

### Best feasible answer

If the MIP reaches its time limit first, the selected store set still satisfies the 50 km requirement, but the true minimum may be smaller.

In that case report:

- best feasible number of stores,
- MIP lower bound,
- relative optimality gap,
- time limit.

This is more rigorous than presenting an unproven incumbent as the minimum.


# TASK 2 — Metric TSP on Selected Store Locations

The great-circle distance matrix is metric.

We use the store locations found in Task 1 as the logistics / inspection nodes.

For the explicit exponential ILP, we use small benchmark subsets because literally enumerating all subtour inequalities becomes impractical as $n$ grows.

The lazy-row formulation can then be run on the full selected-store set.


In [ ]:
# Cell 16 — Pairwise Haversine distance matrix for selected stores

store_coordinates_rad = np.radians(
    selected_stores[
        ["latitude", "longitude"]
    ].to_numpy(dtype=float)
)

store_distance_matrix = (
    haversine_distances(
        store_coordinates_rad
    )
    * EARTH_RADIUS_KM
)

np.fill_diagonal(
    store_distance_matrix,
    0.0
)

print(
    "Selected store locations for TSP:",
    len(selected_stores)
)

print(
    "Maximum pairwise store distance:",
    round(
        store_distance_matrix.max(),
        3
    ),
    "km"
)


In [ ]:
# Cell 17 — Numerical metric sanity check

def sampled_triangle_inequality_check(
    distance_matrix,
    n_checks=20000,
    seed=42
):
    rng = np.random.default_rng(seed)
    n = distance_matrix.shape[0]

    worst_violation = -np.inf
    violation_count = 0

    for _ in range(n_checks):
        i, j, k = rng.integers(
            0,
            n,
            size=3
        )

        lhs = distance_matrix[i, k]
        rhs = (
            distance_matrix[i, j]
            + distance_matrix[j, k]
        )

        violation = lhs - rhs
        worst_violation = max(
            worst_violation,
            violation
        )

        if violation > 1e-8:
            violation_count += 1

    return violation_count, worst_violation


metric_violations, worst_metric_violation = (
    sampled_triangle_inequality_check(
        store_distance_matrix
    )
)

print(
    "Sampled triangle-inequality violations:",
    metric_violations
)

print(
    "Worst numerical lhs-rhs:",
    worst_metric_violation
)


In [ ]:
# Cell 18 — TSP helper: edge indexing

def build_complete_undirected_edges(n):
    return [
        (i, j)
        for i in range(n)
        for j in range(i + 1, n)
    ]


def edge_cost_vector(distance_matrix):
    edges = build_complete_undirected_edges(
        distance_matrix.shape[0]
    )

    costs = np.array(
        [
            distance_matrix[i, j]
            for i, j in edges
        ],
        dtype=float
    )

    return edges, costs


In [ ]:
# Cell 19 — TSP helper: degree-constraint matrix

def degree_constraint_matrix(n, edges):
    row = []
    col = []
    data = []

    for edge_index, (i, j) in enumerate(edges):
        row.extend([i, j])
        col.extend(
            [edge_index, edge_index]
        )
        data.extend([1.0, 1.0])

    return coo_matrix(
        (data, (row, col)),
        shape=(n, len(edges))
    ).tocsr()


In [ ]:
# Cell 20 — TSP helper: convert selected edges into cycles/components

def selected_components(
    x,
    edges,
    n,
    threshold=0.5
):
    adjacency = {
        i: []
        for i in range(n)
    }

    selected_edge_list = []

    for value, (i, j) in zip(x, edges):
        if value > threshold:
            adjacency[i].append(j)
            adjacency[j].append(i)
            selected_edge_list.append(
                (i, j)
            )

    components = []
    seen = set()

    for start in range(n):
        if start in seen:
            continue

        stack = [start]
        seen.add(start)
        component = []

        while stack:
            u = stack.pop()
            component.append(u)

            for v in adjacency[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)

        components.append(
            sorted(component)
        )

    return components, selected_edge_list


In [ ]:
# Cell 21 — TSP helper: recover Hamiltonian tour from selected edges

def recover_tour(
    selected_edges,
    n
):
    adjacency = {
        i: []
        for i in range(n)
    }

    for i, j in selected_edges:
        adjacency[i].append(j)
        adjacency[j].append(i)

    if not all(
        len(adjacency[i]) == 2
        for i in range(n)
    ):
        raise ValueError(
            "Selected solution is not a 2-regular tour."
        )

    tour = [0]
    previous = None
    current = 0

    while True:
        neighbors = adjacency[current]

        next_node = (
            neighbors[0]
            if neighbors[0] != previous
            else neighbors[1]
        )

        if next_node == tour[0]:
            tour.append(next_node)
            break

        tour.append(next_node)
        previous, current = (
            current,
            next_node
        )

        if len(tour) > n + 1:
            raise RuntimeError(
                "Tour reconstruction exceeded expected length."
            )

    if len(set(tour[:-1])) != n:
        raise ValueError(
            "Recovered cycle does not visit every node exactly once."
        )

    return tour


In [ ]:
# Cell 22 — Exact TSP with explicit exponential subtour constraints

def solve_tsp_explicit(
    distance_matrix,
    time_limit=TSP_TIME_LIMIT_SECONDS
):
    n = distance_matrix.shape[0]

    if n > EXPLICIT_TSP_MAX_N:
        raise ValueError(
            f"Explicit exponential formulation is intentionally "
            f"restricted to n <= {EXPLICIT_TSP_MAX_N}."
        )

    edges, costs = edge_cost_vector(
        distance_matrix
    )

    m = len(edges)

    degree_matrix = (
        degree_constraint_matrix(
            n,
            edges
        )
    )

    matrices = [degree_matrix]
    lower_bounds = [
        np.full(n, 2.0)
    ]
    upper_bounds = [
        np.full(n, 2.0)
    ]

    # Generate SECs.
    # With degree constraints, complement cuts are redundant.
    # Enumerating up to floor(n/2) retains a full exact SEC family
    # without unnecessary complement duplicates.
    sec_count = 0

    for subset_size in range(
        2,
        n // 2 + 1
    ):
        for subset in itertools.combinations(
            range(n),
            subset_size
        ):
            subset_set = set(subset)

            internal_edges = [
                edge_index
                for edge_index, (i, j)
                in enumerate(edges)
                if i in subset_set
                and j in subset_set
            ]

            if not internal_edges:
                continue

            sec_row = coo_matrix(
                (
                    np.ones(
                        len(internal_edges)
                    ),
                    (
                        np.zeros(
                            len(internal_edges),
                            dtype=int
                        ),
                        internal_edges
                    )
                ),
                shape=(1, m)
            ).tocsr()

            matrices.append(sec_row)
            lower_bounds.append(
                np.array([-np.inf])
            )
            upper_bounds.append(
                np.array(
                    [subset_size - 1.0]
                )
            )

            sec_count += 1

    A = vstack(
        matrices
    ).tocsr()

    lb = np.concatenate(
        lower_bounds
    )

    ub = np.concatenate(
        upper_bounds
    )

    start = time.perf_counter()

    result = milp(
        c=costs,
        integrality=np.ones(
            m,
            dtype=int
        ),
        bounds=Bounds(
            np.zeros(m),
            np.ones(m)
        ),
        constraints=LinearConstraint(
            A,
            lb=lb,
            ub=ub
        ),
        options={
            "time_limit": time_limit,
            "mip_rel_gap": TSP_MIP_GAP_TARGET,
            "presolve": True
        }
    )

    runtime = (
        time.perf_counter()
        - start
    )

    tour = None
    selected_edges = None

    if result.x is not None:
        components, selected_edges = (
            selected_components(
                result.x,
                edges,
                n
            )
        )

        if len(components) == 1:
            tour = recover_tour(
                selected_edges,
                n
            )

    return {
        "method":
            "Explicit exponential SEC ILP",
        "n":
            n,
        "objective":
            (
                float(result.fun)
                if result.fun is not None
                else np.nan
            ),
        "runtime_seconds":
            runtime,
        "success":
            bool(result.success),
        "message":
            result.message,
        "mip_gap":
            getattr(
                result,
                "mip_gap",
                np.nan
            ),
        "sec_count":
            sec_count,
        "tour":
            tour,
        "selected_edges":
            selected_edges,
        "raw_result":
            result
    }


In [ ]:
# Cell 23 — Exact TSP with lazy row generation

def solve_tsp_lazy_rows(
    distance_matrix,
    time_limit=TSP_TIME_LIMIT_SECONDS,
    max_rounds=100
):
    n = distance_matrix.shape[0]

    edges, costs = edge_cost_vector(
        distance_matrix
    )

    m = len(edges)

    degree_matrix = (
        degree_constraint_matrix(
            n,
            edges
        )
    )

    generated_subsets = set()
    cut_rows = []

    iteration_history = []

    overall_start = time.perf_counter()
    final_result = None
    final_tour = None
    final_selected_edges = None

    for round_number in range(
        1,
        max_rounds + 1
    ):
        elapsed = (
            time.perf_counter()
            - overall_start
        )

        remaining_time = max(
            1.0,
            time_limit - elapsed
        )

        matrices = [degree_matrix]
        lower_bounds = [
            np.full(n, 2.0)
        ]
        upper_bounds = [
            np.full(n, 2.0)
        ]

        if cut_rows:
            matrices.extend(cut_rows)
            lower_bounds.extend(
                [
                    np.array([-np.inf])
                    for _ in cut_rows
                ]
            )

            upper_bounds.extend(
                [
                    np.array(
                        [len(S) - 1.0]
                    )
                    for S in generated_subsets
                ]
            )

        A = vstack(
            matrices
        ).tocsr()

        lb = np.concatenate(
            lower_bounds
        )

        ub = np.concatenate(
            upper_bounds
        )

        result = milp(
            c=costs,
            integrality=np.ones(
                m,
                dtype=int
            ),
            bounds=Bounds(
                np.zeros(m),
                np.ones(m)
            ),
            constraints=LinearConstraint(
                A,
                lb=lb,
                ub=ub
            ),
            options={
                "time_limit": remaining_time,
                "mip_rel_gap": TSP_MIP_GAP_TARGET,
                "presolve": True
            }
        )

        final_result = result

        if result.x is None:
            break

        components, selected_edges = (
            selected_components(
                result.x,
                edges,
                n
            )
        )

        iteration_history.append({
            "round": round_number,
            "objective": float(result.fun),
            "components": len(components),
            "component_sizes": [
                len(component)
                for component
                in components
            ],
            "cuts_total": len(
                generated_subsets
            )
        })

        if len(components) == 1:
            final_selected_edges = (
                selected_edges
            )

            final_tour = recover_tour(
                selected_edges,
                n
            )

            break

        new_cuts_added = 0

        for component in components:
            if (
                len(component) < 2
                or len(component) > n - 2
            ):
                continue

            subset_tuple = tuple(
                sorted(component)
            )

            if subset_tuple in generated_subsets:
                continue

            subset_set = set(
                subset_tuple
            )

            internal_edge_indices = [
                edge_index
                for edge_index, (i, j)
                in enumerate(edges)
                if i in subset_set
                and j in subset_set
            ]

            row = coo_matrix(
                (
                    np.ones(
                        len(internal_edge_indices)
                    ),
                    (
                        np.zeros(
                            len(internal_edge_indices),
                            dtype=int
                        ),
                        internal_edge_indices
                    )
                ),
                shape=(1, m)
            ).tocsr()

            generated_subsets.add(
                subset_tuple
            )

            cut_rows.append(row)
            new_cuts_added += 1

        if new_cuts_added == 0:
            break

        if (
            time.perf_counter()
            - overall_start
        ) >= time_limit:
            break

    runtime = (
        time.perf_counter()
        - overall_start
    )

    return {
        "method":
            "Lazy row generation",
        "n":
            n,
        "objective":
            (
                float(final_result.fun)
                if final_result is not None
                and final_result.fun is not None
                else np.nan
            ),
        "runtime_seconds":
            runtime,
        "success":
            bool(
                final_result is not None
                and final_result.success
                and final_tour is not None
            ),
        "message":
            (
                final_result.message
                if final_result is not None
                else "No MILP result"
            ),
        "mip_gap":
            (
                getattr(
                    final_result,
                    "mip_gap",
                    np.nan
                )
                if final_result is not None
                else np.nan
            ),
        "cuts_generated":
            len(generated_subsets),
        "rounds":
            len(iteration_history),
        "history":
            pd.DataFrame(
                iteration_history
            ),
        "tour":
            final_tour,
        "selected_edges":
            final_selected_edges,
        "raw_result":
            final_result
    }


## Important note about "lazy" constraints

SciPy/HiGHS does not expose a callback interface identical to commercial MIP solvers' native lazy-constraint callbacks.

Therefore this notebook implements **row generation explicitly**:

1. solve current ILP,
2. identify subtours,
3. add violated subtour constraints,
4. resolve.

Algorithmically, this demonstrates the same key idea: do not generate the exponential subtour family unless the solver actually needs those rows.


In [ ]:
# Cell 24 — Choose reproducible benchmark subsets

# Prefer selected stores with larger recorded populations.
# Population is used only to make the subset choice reproducible,
# not as part of the TSP objective.

selected_for_benchmark = (
    selected_stores
    .assign(
        population_numeric=pd.to_numeric(
            selected_stores["population"],
            errors="coerce"
        ).fillna(0)
    )
    .sort_values(
        [
            "population_numeric",
            "name"
        ],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

candidate_sizes = [
    n
    for n in [8, 10, 12, 14]
    if n <= len(selected_for_benchmark)
]

print("Benchmark sizes:", candidate_sizes)


In [ ]:
# Cell 25 — Helper to build distance matrix for a benchmark subset

def benchmark_distance_matrix(n):
    subset = (
        selected_for_benchmark
        .iloc[:n]
        .copy()
    )

    coords_rad = np.radians(
        subset[
            ["latitude", "longitude"]
        ].to_numpy(dtype=float)
    )

    distance_matrix = (
        haversine_distances(
            coords_rad
        )
        * EARTH_RADIUS_KM
    )

    np.fill_diagonal(
        distance_matrix,
        0.0
    )

    return subset, distance_matrix


In [ ]:
# Cell 26 — Run explicit exponential ILP and lazy row generation

exact_comparison_rows = []
exact_solutions = {}

for n in candidate_sizes:
    print("=" * 72)
    print("TSP benchmark size:", n)

    subset, D = (
        benchmark_distance_matrix(n)
    )

    explicit = solve_tsp_explicit(
        D
    )

    lazy = solve_tsp_lazy_rows(
        D
    )

    exact_solutions[n] = {
        "subset": subset,
        "distance_matrix": D,
        "explicit": explicit,
        "lazy": lazy
    }

    exact_comparison_rows.append({
        "n": n,
        "explicit_objective_km":
            explicit["objective"],
        "explicit_runtime_sec":
            explicit[
                "runtime_seconds"
            ],
        "explicit_SEC_count":
            explicit["sec_count"],
        "explicit_success":
            explicit["success"],
        "lazy_objective_km":
            lazy["objective"],
        "lazy_runtime_sec":
            lazy[
                "runtime_seconds"
            ],
        "lazy_cuts_generated":
            lazy["cuts_generated"],
        "lazy_rounds":
            lazy["rounds"],
        "lazy_success":
            lazy["success"],
        "objective_abs_difference":
            abs(
                explicit["objective"]
                - lazy["objective"]
            )
    })


exact_comparison = pd.DataFrame(
    exact_comparison_rows
)

display(exact_comparison)


In [ ]:
# Cell 27 — Check whether the two exact formulations agree

if len(exact_comparison) > 0:
    exact_agreement = (
        exact_comparison[
            "objective_abs_difference"
        ]
        .fillna(np.inf)
        .max()
        < 1e-5
    )

    print(
        "Explicit ILP and lazy row generation "
        "agree on all tested exact objectives:",
        bool(exact_agreement)
    )

    display(
        exact_comparison[
            [
                "n",
                "explicit_objective_km",
                "lazy_objective_km",
                "explicit_runtime_sec",
                "lazy_runtime_sec",
                "explicit_SEC_count",
                "lazy_cuts_generated",
                "lazy_rounds"
            ]
        ]
    )
else:
    exact_agreement = False
    print("No benchmark instances were available.")


In [ ]:
# Cell 28 — Plot constraint growth

if len(exact_comparison) > 0:
    plt.figure(figsize=(8, 5))

    plt.semilogy(
        exact_comparison["n"],
        exact_comparison[
            "explicit_SEC_count"
        ],
        marker="o",
        label="Explicit SECs"
    )

    plt.semilogy(
        exact_comparison["n"],
        np.maximum(
            exact_comparison[
                "lazy_cuts_generated"
            ],
            1
        ),
        marker="o",
        label="Lazy rows actually added"
    )

    plt.xlabel("Number of TSP nodes")
    plt.ylabel("Number of subtour constraints (log scale)")
    plt.title(
        "Explicit Exponential Constraints vs Lazy Row Generation"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 29 — Plot exact-method runtime comparison

if len(exact_comparison) > 0:
    plt.figure(figsize=(8, 5))

    plt.plot(
        exact_comparison["n"],
        exact_comparison[
            "explicit_runtime_sec"
        ],
        marker="o",
        label="Explicit exponential SEC ILP"
    )

    plt.plot(
        exact_comparison["n"],
        exact_comparison[
            "lazy_runtime_sec"
        ],
        marker="o",
        label="Lazy row generation"
    )

    plt.xlabel("Number of TSP nodes")
    plt.ylabel("Runtime (seconds)")
    plt.title("Exact TSP Runtime Comparison")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 30 — Solve the full selected-store TSP with lazy row generation

full_lazy_tsp = solve_tsp_lazy_rows(
    store_distance_matrix,
    time_limit=TSP_TIME_LIMIT_SECONDS
)

print("Full store TSP nodes:", full_lazy_tsp["n"])
print("Solver success:", full_lazy_tsp["success"])
print(
    "Best / optimal tour length:",
    full_lazy_tsp["objective"],
    "km"
)
print(
    "Lazy cuts generated:",
    full_lazy_tsp["cuts_generated"]
)
print(
    "Row-generation rounds:",
    full_lazy_tsp["rounds"]
)
print(
    "Runtime:",
    round(
        full_lazy_tsp[
            "runtime_seconds"
        ],
        4
    ),
    "seconds"
)

display(
    full_lazy_tsp["history"]
)


In [ ]:
# Cell 31 — Plot the full exact/lazy TSP route when available

def plot_tour(
    points_df,
    tour,
    title
):
    if tour is None:
        print("No complete tour is available.")
        return

    x = points_df[
        "longitude"
    ].to_numpy()

    y = points_df[
        "latitude"
    ].to_numpy()

    plt.figure(figsize=(9, 8))

    plt.scatter(
        x,
        y,
        s=40
    )

    for k in range(
        len(tour) - 1
    ):
        i = tour[k]
        j = tour[k + 1]

        plt.plot(
            [x[i], x[j]],
            [y[i], y[j]],
            linewidth=1
        )

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_tour(
    selected_stores,
    full_lazy_tsp["tour"],
    "Full Selected-Store TSP — Lazy Row Generation"
)


# TASK 3 — Christofides Approximation

We now solve the same metric instances using Christofides' algorithm.

The implementation is built from:

- NetworkX minimum spanning tree,
- minimum-weight perfect matching,
- Eulerian circuit,
- shortcutting repeated vertices.


In [ ]:
# Cell 32 — Christofides algorithm implementation

def christofides_tour(
    distance_matrix
):
    n = distance_matrix.shape[0]

    complete_graph = nx.Graph()

    for i in range(n):
        for j in range(
            i + 1,
            n
        ):
            complete_graph.add_edge(
                i,
                j,
                weight=float(
                    distance_matrix[i, j]
                )
            )

    mst = nx.minimum_spanning_tree(
        complete_graph,
        weight="weight"
    )

    odd_vertices = [
        node
        for node, degree
        in mst.degree()
        if degree % 2 == 1
    ]

    odd_graph = nx.Graph()

    for i, j in itertools.combinations(
        odd_vertices,
        2
    ):
        odd_graph.add_edge(
            i,
            j,
            weight=float(
                distance_matrix[i, j]
            )
        )

    matching = (
        nx.algorithms.matching
        .min_weight_matching(
            odd_graph,
            weight="weight"
        )
    )

    multigraph = nx.MultiGraph()

    multigraph.add_nodes_from(
        complete_graph.nodes()
    )

    for i, j, data in mst.edges(
        data=True
    ):
        multigraph.add_edge(
            i,
            j,
            weight=data["weight"]
        )

    for pair in matching:
        i, j = tuple(pair)

        multigraph.add_edge(
            i,
            j,
            weight=float(
                distance_matrix[i, j]
            )
        )

    if not nx.is_eulerian(
        multigraph
    ):
        raise RuntimeError(
            "Christofides multigraph is not Eulerian."
        )

    euler_edges = list(
        nx.eulerian_circuit(
            multigraph
        )
    )

    if not euler_edges:
        raise RuntimeError(
            "Eulerian circuit was empty."
        )

    start_node = euler_edges[0][0]

    euler_vertex_sequence = [
        start_node
    ] + [
        edge[1]
        for edge in euler_edges
    ]

    shortcut_tour = []
    visited = set()

    for node in euler_vertex_sequence:
        if node not in visited:
            shortcut_tour.append(node)
            visited.add(node)

    shortcut_tour.append(
        shortcut_tour[0]
    )

    length = sum(
        distance_matrix[
            shortcut_tour[k],
            shortcut_tour[k + 1]
        ]
        for k in range(
            len(shortcut_tour) - 1
        )
    )

    mst_length = sum(
        data["weight"]
        for _, _, data
        in mst.edges(data=True)
    )

    matching_length = sum(
        distance_matrix[i, j]
        for i, j in matching
    )

    return {
        "tour":
            shortcut_tour,
        "length_km":
            float(length),
        "mst":
            mst,
        "mst_length_km":
            float(mst_length),
        "odd_vertices":
            odd_vertices,
        "matching":
            matching,
        "matching_length_km":
            float(matching_length)
    }


In [ ]:
# Cell 33 — Compare Christofides with exact ILP benchmarks

christofides_rows = []
christofides_solutions = {}

for n in candidate_sizes:
    subset = (
        exact_solutions[n][
            "subset"
        ]
    )

    D = (
        exact_solutions[n][
            "distance_matrix"
        ]
    )

    start = time.perf_counter()

    christofides = (
        christofides_tour(D)
    )

    runtime = (
        time.perf_counter()
        - start
    )

    exact_objective = (
        exact_solutions[n][
            "lazy"
        ]["objective"]
    )

    ratio = (
        christofides[
            "length_km"
        ]
        / exact_objective
    )

    christofides_solutions[n] = (
        christofides
    )

    christofides_rows.append({
        "n":
            n,
        "exact_optimum_km":
            exact_objective,
        "christofides_km":
            christofides[
                "length_km"
            ],
        "approximation_ratio":
            ratio,
        "excess_over_optimum_pct":
            100.0 * (
                ratio - 1.0
            ),
        "christofides_runtime_sec":
            runtime,
        "theoretical_1_5_bound_satisfied":
            bool(
                ratio <= 1.5 + 1e-8
            )
    })


christofides_comparison = (
    pd.DataFrame(
        christofides_rows
    )
)

display(
    christofides_comparison
)


In [ ]:
# Cell 34 — Christofides summary statistics

if len(christofides_comparison) > 0:
    print(
        "Mean approximation ratio:",
        round(
            christofides_comparison[
                "approximation_ratio"
            ].mean(),
            6
        )
    )

    print(
        "Worst observed approximation ratio:",
        round(
            christofides_comparison[
                "approximation_ratio"
            ].max(),
            6
        )
    )

    print(
        "Best observed approximation ratio:",
        round(
            christofides_comparison[
                "approximation_ratio"
            ].min(),
            6
        )
    )

    print(
        "All tested ratios satisfy 1.5 bound:",
        bool(
            christofides_comparison[
                "theoretical_1_5_bound_satisfied"
            ].all()
        )
    )


In [ ]:
# Cell 35 — Christofides on the full store set

christofides_full_start = (
    time.perf_counter()
)

full_christofides = (
    christofides_tour(
        store_distance_matrix
    )
)

full_christofides_runtime = (
    time.perf_counter()
    - christofides_full_start
)

print(
    "Full Christofides tour length:",
    round(
        full_christofides[
            "length_km"
        ],
        3
    ),
    "km"
)

print(
    "Christofides runtime:",
    round(
        full_christofides_runtime,
        5
    ),
    "seconds"
)

if (
    full_lazy_tsp["tour"]
    is not None
    and np.isfinite(
        full_lazy_tsp["objective"]
    )
):
    full_christofides_ratio = (
        full_christofides[
            "length_km"
        ]
        / full_lazy_tsp[
            "objective"
        ]
    )

    print(
        "Ratio vs full lazy-row TSP solution:",
        round(
            full_christofides_ratio,
            6
        )
    )
else:
    full_christofides_ratio = np.nan

    print(
        "No complete full exact/lazy tour was available "
        "for a full-set ratio."
    )


In [ ]:
# Cell 36 — Plot full Christofides route

plot_tour(
    selected_stores,
    full_christofides["tour"],
    "Full Selected-Store TSP — Christofides"
)


In [ ]:
# Cell 37 — Exact vs Christofides route length plot

if len(christofides_comparison) > 0:
    plt.figure(figsize=(8, 5))

    plt.plot(
        christofides_comparison["n"],
        christofides_comparison[
            "exact_optimum_km"
        ],
        marker="o",
        label="Exact TSP"
    )

    plt.plot(
        christofides_comparison["n"],
        christofides_comparison[
            "christofides_km"
        ],
        marker="o",
        label="Christofides"
    )

    plt.xlabel("Number of TSP nodes")
    plt.ylabel("Tour length (km)")
    plt.title(
        "Christofides vs Exact Metric TSP"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 38 — Approximation-ratio plot

if len(christofides_comparison) > 0:
    plt.figure(figsize=(8, 4))

    plt.plot(
        christofides_comparison["n"],
        christofides_comparison[
            "approximation_ratio"
        ],
        marker="o"
    )

    plt.axhline(
        1.0,
        linestyle="--",
        label="Optimal"
    )

    plt.axhline(
        1.5,
        linestyle="--",
        label="Christofides theoretical bound"
    )

    plt.xlabel("Number of TSP nodes")
    plt.ylabel("Christofides / optimum")
    plt.title(
        "Observed Christofides Approximation Ratio"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


# Research Interpretation

### Facility-location result

The set-covering model answers the expansion question directly for the supplied Polish populated-place data.

The most important distinction is whether the MIP has:

- **proved optimality**, or
- merely found a strong feasible incumbent before the time limit.

### Exact TSP formulations

The explicit subtour model demonstrates why the textbook formulation is difficult to scale: the number of subtour inequalities grows exponentially.

Lazy row generation avoids creating most of those rows and adds only constraints corresponding to subtours actually encountered.

### Christofides

Christofides is not exact, but on metric instances it provides a strong polynomial-time approximation.

The empirical comparison should discuss:

- tour-length gap,
- approximation ratio,
- runtime,
- scalability.


In [ ]:
# Cell 39 — Build final answer table for the three assignment tasks

task_summary_rows = []

task_summary_rows.append({
    "Task":
        "Task 1 — 50 km store coverage",
    "Main_result":
        f"{len(selected_stores)} stores",
    "Status":
        (
            "Proven optimal"
            if facility_proven_optimal
            else "Best feasible solution; optimality not proven"
        ),
    "Supporting_metric":
        (
            f"max nearest-store distance = "
            f"{coverage_validation['nearest_store_distance_km'].max():.3f} km"
        )
})

if len(exact_comparison) > 0:
    largest_exact_row = (
        exact_comparison
        .sort_values("n")
        .iloc[-1]
    )

    task_summary_rows.append({
        "Task":
            "Task 2 — exact metric TSP",
        "Main_result":
            (
                f"n={int(largest_exact_row['n'])}: "
                f"{largest_exact_row['lazy_objective_km']:.3f} km"
            ),
        "Status":
            (
                "Explicit SEC and lazy-row objectives agree"
                if exact_agreement
                else "Check exact-method agreement"
            ),
        "Supporting_metric":
            (
                f"explicit SECs = "
                f"{int(largest_exact_row['explicit_SEC_count'])}; "
                f"lazy rows = "
                f"{int(largest_exact_row['lazy_cuts_generated'])}"
            )
    })

if len(christofides_comparison) > 0:
    worst_ratio_row = (
        christofides_comparison
        .sort_values(
            "approximation_ratio",
            ascending=False
        )
        .iloc[0]
    )

    task_summary_rows.append({
        "Task":
            "Task 3 — Christofides",
        "Main_result":
            (
                f"worst tested ratio = "
                f"{worst_ratio_row['approximation_ratio']:.4f}"
            ),
        "Status":
            (
                "1.5 bound satisfied on all tested instances"
                if christofides_comparison[
                    "theoretical_1_5_bound_satisfied"
                ].all()
                else "Unexpected ratio; inspect implementation"
            ),
        "Supporting_metric":
            (
                f"mean ratio = "
                f"{christofides_comparison['approximation_ratio'].mean():.4f}"
            )
    })


task_summary = pd.DataFrame(
    task_summary_rows
)

display(task_summary)


In [ ]:
# Cell 40 — One-cell final question-and-answer report

from IPython.display import Markdown, display

report = []

report.append(
    "# Final Assignment Answers"
)

report.append(
    "## 1. How many Steamboat Willie's stores are required "
    "to place every Polish populated place within 50 km?"
)

report.append(
    f"The best solution used by the notebook opens "
    f"**{len(selected_stores)} stores**."
)

report.append(
    f"All **{len(poland):,}** Polish populated places in the "
    f"supplied dataset are covered within 50 km."
)

report.append(
    f"The largest observed distance from a populated place "
    f"to its nearest selected store is "
    f"**{coverage_validation['nearest_store_distance_km'].max():.3f} km**."
)

if facility_proven_optimal:
    report.append(
        "The MIP solver **proved this store count optimal** "
        "for the supplied-data model."
    )
else:
    report.append(
        "The MIP solver did **not** prove optimality within the "
        "configured time limit, so this should be reported as the "
        "**best feasible store count found**, not as a mathematically "
        "proven minimum."
    )

    if np.isfinite(facility_lower_bound):
        report.append(
            f"The solver's lower bound is "
            f"**{facility_lower_bound:.3f} stores**."
        )

    if np.isfinite(facility_gap):
        report.append(
            f"The final relative MIP gap is "
            f"**{100.0 * facility_gap:.2f}%**."
        )

report.append(
    "## 2. What did the exact Metric TSP experiments show?"
)

if len(exact_comparison) > 0:
    report.append(
        f"The explicit exponential subtour ILP and the lazy-row "
        f"formulation were tested on instances of size "
        f"**{', '.join(map(str, candidate_sizes))}**."
    )

    if exact_agreement:
        report.append(
            "The two exact formulations produced the "
            "**same optimal tour length on every tested benchmark instance**."
        )
    else:
        report.append(
            "At least one tested instance did not show matching "
            "exact objectives, so those results should be inspected."
        )

    biggest = (
        exact_comparison
        .sort_values("n")
        .iloc[-1]
    )

    report.append(
        f"For the largest explicit benchmark, $n={int(biggest['n'])}$, "
        f"the optimal tour length was approximately "
        f"**{biggest['lazy_objective_km']:.3f} km**."
    )

    report.append(
        f"The explicit formulation generated "
        f"**{int(biggest['explicit_SEC_count']):,}** subtour rows, "
        f"whereas lazy row generation needed only "
        f"**{int(biggest['lazy_cuts_generated']):,}** generated cuts."
    )

report.append(
    "The direct exponential formulation is therefore useful for "
    "demonstrating the textbook ILP, but lazy row generation is "
    "far more attractive as the TSP grows because it avoids "
    "materializing most subtour constraints."
)

report.append(
    "## 3. What happened on the full selected-store TSP?"
)

if full_lazy_tsp["tour"] is not None:
    report.append(
        f"The lazy-row formulation produced a complete tour through "
        f"all **{len(selected_stores)}** selected store locations "
        f"with length **{full_lazy_tsp['objective']:.3f} km**."
    )

    report.append(
        f"It generated **{full_lazy_tsp['cuts_generated']}** "
        f"subtour cuts over **{full_lazy_tsp['rounds']}** "
        f"row-generation round(s)."
    )
else:
    report.append(
        "The full selected-store lazy-row model did not complete "
        "within the configured conditions. Increase the TSP time "
        "limit before reporting an exact full-set result."
    )

report.append(
    "## 4. How did Christofides compare with the exact TSP?"
)

if len(christofides_comparison) > 0:
    mean_ratio = (
        christofides_comparison[
            "approximation_ratio"
        ].mean()
    )

    worst_ratio = (
        christofides_comparison[
            "approximation_ratio"
        ].max()
    )

    mean_excess = (
        christofides_comparison[
            "excess_over_optimum_pct"
        ].mean()
    )

    report.append(
        f"Across the exact benchmark instances, Christofides had a "
        f"mean approximation ratio of **{mean_ratio:.4f}**."
    )

    report.append(
        f"The worst observed ratio was **{worst_ratio:.4f}**."
    )

    report.append(
        f"On average, the Christofides tours were "
        f"**{mean_excess:.2f}% longer** than the exact optimum."
    )

    all_bound = bool(
        christofides_comparison[
            "theoretical_1_5_bound_satisfied"
        ].all()
    )

    report.append(
        "All tested Christofides solutions "
        + (
            "**satisfied the theoretical 1.5 metric-TSP bound**."
            if all_bound
            else "did **not** satisfy the expected 1.5 bound; inspect the implementation."
        )
    )

report.append(
    "## 5. What is the practical OR conclusion?"
)

report.append(
    "The three tasks illustrate different layers of Operations Research. "
    "The covering model determines **where stores should be opened** "
    "to satisfy a geographic service requirement. The exact TSP models "
    "determine **how a route through chosen locations can be optimized**. "
    "The Christofides algorithm demonstrates the trade-off between "
    "**exact optimality and computational scalability**."
)

report.append(
    "For business planning, the 50 km covering result should be treated "
    "as a geographic baseline. A real expansion model would also include "
    "population demand, expected revenue, road travel time, real-estate "
    "cost, competition, store capacity, and regional operating constraints."
)

display(
    Markdown(
        "\n\n".join(report)
    )
)


In [ ]:
# Cell 41 — Save important outputs

poland.to_csv(
    OUTPUT_DIR
    / "poland_populated_places.csv",
    index=False
)

selected_stores.to_csv(
    OUTPUT_DIR
    / "selected_store_locations.csv",
    index=False
)

coverage_validation.to_csv(
    OUTPUT_DIR
    / "coverage_validation.csv",
    index=False
)

exact_comparison.to_csv(
    OUTPUT_DIR
    / "tsp_exact_method_comparison.csv",
    index=False
)

christofides_comparison.to_csv(
    OUTPUT_DIR
    / "christofides_vs_exact.csv",
    index=False
)

task_summary.to_csv(
    OUTPUT_DIR
    / "assignment_task_summary.csv",
    index=False
)

if full_lazy_tsp["tour"] is not None:
    pd.DataFrame({
        "tour_order":
            np.arange(
                len(
                    full_lazy_tsp[
                        "tour"
                    ]
                )
            ),
        "store_local_index":
            full_lazy_tsp[
                "tour"
            ]
    }).to_csv(
        OUTPUT_DIR
        / "full_lazy_tsp_tour.csv",
        index=False
    )

pd.DataFrame({
    "tour_order":
        np.arange(
            len(
                full_christofides[
                    "tour"
                ]
            )
        ),
    "store_local_index":
        full_christofides[
            "tour"
        ]
}).to_csv(
    OUTPUT_DIR
    / "full_christofides_tour.csv",
    index=False
)

print("Saved output files:")

for path in sorted(
    OUTPUT_DIR.glob("*.csv")
):
    print("-", path.name)


# Report-Ready Discussion Points

### Task 1 — Facility location

Discuss:

- number of Polish populated places represented,
- 50 km service requirement,
- set-cover formulation,
- exact versus best-feasible status,
- maximum nearest-store distance,
- geographic distribution of selected stores.

### Task 2 — Exact TSP

Discuss:

- metric Haversine distances,
- degree constraints,
- subtour problem,
- exponential growth in explicit subtour constraints,
- equality of the explicit and lazy-row optimal objectives,
- runtime and constraint-count differences.

### Task 3 — Christofides

Discuss:

- MST,
- odd-degree matching,
- Eulerian circuit,
- shortcutting,
- exact optimum comparison,
- observed approximation ratio,
- theoretical 1.5 bound,
- speed versus optimality trade-off.


# Limitations

The analysis is intentionally focused on Operations Research methodology.

### Location-model limitations

A 50 km great-circle radius is not the same as:

- 50 km road distance,
- 50 km driving distance,
- a 50-minute catchment.

The model also ignores:

- demand intensity,
- store capacity,
- construction cost,
- property availability,
- competitors,
- highways and rivers,
- operating cost,
- expected revenue.

### TSP limitations

The TSP assumes:

- symmetric travel distances,
- one vehicle,
- no time windows,
- no service times,
- no vehicle capacity,
- no depots other than the cycle start/end convention.

A logistics extension could use:

- Vehicle Routing Problem,
- Capacitated VRP,
- VRP with Time Windows,
- Location-Routing Problem.


# Strong Next Research Extension

A particularly strong follow-up project would integrate Tasks 1 and 2 into a **Location-Routing Problem (LRP)**.

Instead of deciding store locations first and routing second, jointly optimize:

$$\text{facility opening cost}
+
\text{routing cost}
$$

subject to:

- coverage requirements,
- facility capacities,
- customer assignment,
- vehicle routes,
- budget constraints.

That moves the project from separate facility-location and TSP exercises into a richer OR research model.


# Suggested Research Title

## **Geographic Facility Location and Metric Routing in Poland: Set Covering, Exact TSP, Lazy Row Generation, and Christofides Approximation**

Alternative:

## **From Market Coverage to Distribution Routing: An Operations Research Study of Facility Location and Metric TSP in Poland**

More technical:

## **Minimum Covering and Metric Traveling Salesman Optimization on Geospatial Population Data**


# CV Wording — Use After Running and Verifying the Results

### Facility Location and Transportation Network Optimization

**Python, SciPy/HiGHS, Integer Programming, Set Covering, TSP, Graph Algorithms, NetworkX**

Developed an Operations Research framework for geospatial market expansion in Poland, formulating a 50 km minimum set-covering model for candidate store locations and exact metric TSP models using explicit subtour-elimination constraints and iterative lazy row generation; implemented Christofides' approximation algorithm and benchmarked tour quality and computational performance against exact ILP solutions.


# Final Checklist Before Submission

You should be able to answer:

### Task 1
- What exactly counts as a town in the supplied dataset?
- Why is the problem a set-covering problem?
- Why is Haversine distance used?
- Was the reported store count proved optimal?
- Are all demand points within 50 km?

### Task 2
- Why are degree constraints alone insufficient?
- What is a subtour?
- Why are there exponentially many subtour constraints?
- Why does row generation help?
- Did the two exact methods return identical objective values?

### Task 3
- Why does Christofides require a metric?
- What is the role of the MST?
- Why are odd-degree nodes matched?
- Why does shortcutting not increase distance in a metric TSP?
- What approximation ratios did the experiment actually produce?

### Business interpretation
- Why is 50 km great-circle coverage only a preliminary expansion model?
- What variables should be added before making a real store-opening decision?
